# 🥊 AshLane CharacterGen — GPU 3D Character Generation

**One-click:** Runtime → Run all. That's it.

What this does:
1. Checks for a Colab GPU (free T4 works)
2. Installs TripoSR (MIT, image→3D, ~seconds on GPU)
3. Takes a **text prompt** (auto-generates a front-view reference image) **or** an uploaded image
4. Generates a textured 3D character → decimates to game-ready poly count
5. Saves the GLB to Google Drive **and** auto-downloads it

> **Before you run:** Runtime → Change runtime type → **GPU (T4)**. Free tier is fine.
>
> Edit the **CONFIG** cell below to set your prompt / image / batch list, then Run all.


## 1. GPU check

Fails loudly if Colab didn't give us a GPU.

In [ ]:
import torch, sys
print(f"PyTorch {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "\n❌ NO GPU DETECTED.\n"
        "Fix: Runtime → Change runtime type → Hardware accelerator = GPU (T4).\n"
        "Then Runtime → Run all again."
    )
props = torch.cuda.get_device_properties(0)
print(f"✅ GPU: {props.name} — {props.total_memory / 1e9:.1f} GB VRAM")
if props.total_memory < 6e9:
    print("⚠️  <6GB VRAM — TripoSR may OOM. Use a T4 (15GB) runtime.")
DEVICE = "cuda:0"


## 2. Install dependencies

Takes ~2-4 min the first time (TripoSR weights ~1.6GB download on first run).

In [ ]:
# TripoSR (MIT) + text-to-image + mesh tools
%pip install -q git+https://github.com/VAST-AI-Research/TripoSR.git
%pip install -q diffusers transformers accelerate rembg fast-simplification huggingface_hub Pillow trimesh numpy scipy
print("✅ installs done — restart-free (all imports below are fresh)")


## 3. Google Drive (optional but recommended)

Mounts Drive so GLBs persist. If you skip auth, files still download locally.

In [ ]:
DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OK = True
    print("✅ Drive mounted")
except Exception as e:
    print(f"⚠️  Drive not mounted ({e}) — outputs will download locally only")


## 4. CONFIG — ✏️ EDIT ME

Set your prompt (or image), backend, and batch list here. Then Run all.

Tip: use `tools/generative/character/colab-batch.py` to generate the `BATCH` list from a JSON file — paste its output below.

In [ ]:
# ============ CONFIG — EDIT ME ============
INPUT_MODE = "text"   # "text" or "image"

# Single generation (used when BATCH is empty)
PROMPT = "muscular street brawler, green mohawk, black leather jacket, cargo pants"
IMAGE_PATH = ""  # e.g. "/content/ref.png" — or leave empty to get an upload prompt

# Batch mode: list of {"name": ..., "prompt": ...} and/or {"name": ..., "image": ...}
# Paste output of colab-batch.py here. Empty list = single generation above.
BATCH = [
    # {"name": "brawler-01", "prompt": "muscular street brawler, green mohawk, leather jacket"},
    # {"name": "brawler-02", "prompt": "female boxer, purple braids, boxing gloves"},
]

BACKEND = "triposr"      # "triposr" = fast, fits free T4. "trellis2" = better quality, heavier (see §7).
TARGET_FACES = 20000    # game-ready poly target for decimation
OUTPUT_DIR = "/content/ashlane_out"
DRIVE_FOLDER = "/content/drive/MyDrive/AshLane_CharacterGen"
# ==========================================

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
if DRIVE_OK:
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
print(f"Mode={INPUT_MODE} backend={BACKEND} batch_size={len(BATCH)} target_faces={TARGET_FACES}")


## 5. Load models

Loads TripoSR once. SD-Turbo loads only in text mode.

In [ ]:
import torch
from PIL import Image

triposr_model = None
t2i_pipe = None

if BACKEND == "triposr":
    from tsr.system import TSR
    print("Loading TripoSR (stabilityai/TripoSR, ~1.6GB first run)...")
    triposr_model = TSR.from_pretrained(
        "stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt"
    )
    triposr_model.renderer.set_chunk_size(8192)
    triposr_model.to(DEVICE)
    triposr_model.eval()
    print("✅ TripoSR ready")
elif BACKEND == "trellis2":
    print("⏭️  TRELLIS.2 selected — run the optional §7 cells at the bottom instead.")
else:
    raise ValueError(f"Unknown BACKEND={BACKEND!r} (use 'triposr' or 'trellis2')")

if INPUT_MODE == "text" or any("prompt" in b for b in BATCH):
    from diffusers import AutoPipelineForText2Image
    print("Loading SD-Turbo for text→image...")
    t2i_pipe = AutoPipelineForText2Image.from_pretrained(
        "stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16"
    ).to(DEVICE)
    print("✅ SD-Turbo ready")


## 6. Generate

Pipeline per character: **text → front-view reference image → background removal → TripoSR → textured GLB**.
Prompts are auto-enhanced for character work: full-body, front-facing, plain white background (TripoSR quality depends on this).

In [ ]:
import os, shutil, numpy as np, rembg
from PIL import Image

_rembg_session = rembg.new_session()

def enhance_prompt(p: str) -> str:
    return (
        f"{p}, full body 3D game character, T-pose, front facing, "
        f"plain white background, centered, high detail, studio lighting"
    )

def text_to_image(prompt: str) -> Image.Image:
    assert t2i_pipe is not None, "text mode needs the SD-Turbo pipe (cell 5)"
    img = t2i_pipe(
        prompt=enhance_prompt(prompt), num_inference_steps=2, guidance_scale=0.0
    ).images[0]
    return img.convert("RGB")

def get_input_image(item: dict) -> Image.Image:
    if "image" in item and item["image"]:
        return Image.open(item["image"]).convert("RGB")
    if "prompt" in item and item["prompt"]:
        print(f"  🎨 text→image: {item['prompt'][:60]}...")
        return text_to_image(item["prompt"])
    # single-mode fallback
    if INPUT_MODE == "image":
        if IMAGE_PATH:
            return Image.open(IMAGE_PATH).convert("RGB")
        from google.colab import files
        print("📤 Upload a reference image (front-facing, plain background):")
        up = files.upload()
        return Image.open(list(up.keys())[0]).convert("RGB")
    return text_to_image(PROMPT)

def image_to_glb_triposr(image: Image.Image, out_dir: str) -> str:
    from tsr.utils import remove_background, resize_foreground
    img = remove_background(image, _rembg_session)
    img = resize_foreground(img, 0.85)
    # composite RGBA over mid-gray (TripoSR convention)
    arr = np.array(img).astype(np.float32) / 255.0
    rgb = arr[:, :, :3] * arr[:, :, 3:4] + (1 - arr[:, :, 3:4]) * 0.5
    img = Image.fromarray((rgb * 255.0).astype(np.uint8))
    with torch.no_grad():
        if hasattr(triposr_model, "reconstruct"):
            scene = triposr_model.reconstruct([img])
        else:
            scene = triposr_model([img], device=DEVICE)
    os.makedirs(out_dir, exist_ok=True)
    glb_path = os.path.join(out_dir, "raw.glb")
    scene.save_glb(glb_path)
    return glb_path

def generate_one(item: dict) -> str:
    name = item.get("name", "character")
    print(f"\n{'='*50}\n🔨 {name}\n{'='*50}")
    workdir = os.path.join(OUTPUT_DIR, name)
    os.makedirs(workdir, exist_ok=True)
    image = get_input_image(item)
    image.save(os.path.join(workdir, "reference.png"))
    print("  🧊 image→3D (TripoSR)...")
    raw_glb = image_to_glb_triposr(image, workdir)
    print(f"  ✅ raw: {raw_glb}")
    return raw_glb

# Build the job list
if BATCH:
    jobs = BATCH
else:
    jobs = [{"name": "character",
             **({"prompt": PROMPT} if INPUT_MODE == "text" else {"image": IMAGE_PATH})}]

raw_results = {}
if BACKEND == "triposr":
    for job in jobs:
        raw_results[job.get("name", "character")] = generate_one(job)
    torch.cuda.empty_cache()
    print(f"\n✅ Generated {len(raw_results)} raw model(s)")
else:
    print("⏭️  Skipping TripoSR generation — use §7 (TRELLIS.2) cells below.")


## 7. Postprocess → game-ready GLB

Decimates to `TARGET_FACES`, normalizes to 1.8 m tall with feet on the ground, centered. Vertex colors survive decimation.

In [ ]:
import trimesh
import numpy as np

def postprocess_glb(in_path: str, out_path: str, target_faces: int = 20000) -> str:
    scene = trimesh.load(in_path, force="scene")
    parts = []
    for node in scene.graph.nodes_geometry:
        transform, geom_name = scene.graph[node]
        g = scene.geometry[geom_name].copy()
        g.apply_transform(transform)
        parts.append(g)
    mesh = trimesh.util.concatenate(parts) if len(parts) > 1 else parts[0]
    # drop non-triangular junk if present
    if hasattr(mesh, "faces"):
        print(f"  input: {len(mesh.faces)} faces")
        if len(mesh.faces) > target_faces:
            try:
                mesh = mesh.simplify_quadric_decimation(target_faces)  # needs fast-simplification
                print(f"  decimated: {len(mesh.faces)} faces")
            except Exception as e:
                print(f"  ⚠️  decimation failed ({e}) — keeping original")
    # normalize: 1.8m tall, feet at y=0, centered on XZ
    h = mesh.bounds[1][1] - mesh.bounds[0][1]
    if h > 0:
        mesh.apply_scale(1.8 / h)
    mesh.apply_translation([-mesh.centroid[0], -mesh.bounds[0][1], -mesh.centroid[2]])
    mesh.export(out_path)
    print(f"  ✅ game-ready: {out_path}")
    return out_path

final_results = {}
for name, raw in raw_results.items():
    final = os.path.join(OUTPUT_DIR, name, f"{name}.glb")
    print(f"\n⚙️  postprocess {name}...")
    final_results[name] = postprocess_glb(raw, final, TARGET_FACES)
print(f"\n✅ {len(final_results)} game-ready GLB(s)")


## 8. Save to Drive + auto-download

In [ ]:
import shutil
try:
    from google.colab import files as _cfiles
    _has_dl = True
except ImportError:
    _has_dl = False

for name, glb in final_results.items():
    if DRIVE_OK:
        dest = os.path.join(DRIVE_FOLDER, f"{name}.glb")
        shutil.copy2(glb, dest)
        print(f"💾 Drive: {dest}")
    if _has_dl:
        print(f"⬇️  downloading {name}.glb ...")
        _cfiles.download(glb)

print("\n" + "="*50)
print("DONE — files:")
for name, glb in final_results.items():
    print(f"  • {name}: {glb}")
print("="*50)


## 9. (Optional) TRELLIS.2 — higher quality

Set `BACKEND = "trellis2"` in the CONFIG cell, then run these two cells **instead of** §6–§8.
TRELLIS.2 (~3B params, MIT) gives better geometry + PBR textures but needs ~10GB VRAM and ~6GB weights — slow on free Colab, may need Colab Pro for comfort.
Note: `microsoft/TRELLIS.2` weights are access-gated on HuggingFace — accept the terms at https://huggingface.co/microsoft/TRELLIS.2 and run `huggingface-cli login` in a cell before loading the pipeline.


In [ ]:
# Install TRELLIS.2 (heavy — only run if BACKEND == "trellis2")
assert BACKEND == "trellis2", "Set BACKEND='trellis2' in CONFIG first"
%cd /content
!git clone --depth 1 https://github.com/microsoft/TRELLIS.2.git 2>&1 | tail -1
%cd /content/TRELLIS.2
!bash setup.sh 2>&1 | tail -3
print("✅ TRELLIS.2 installed — weights download on first pipeline load (~6GB)")


In [ ]:
# TRELLIS.2 generation (text and/or image → mesh → GLB)
import sys
sys.path.insert(0, "/content/TRELLIS.2")
import torch
from PIL import Image

# NOTE: exact import names follow the TRELLIS.2 repo layout;
# if they drift, check https://github.com/microsoft/TRELLIS.2 for the current API.
from trellis2 import Trellis2Pipeline

print("Loading TRELLIS.2...")
t2_pipe = Trellis2Pipeline.from_pretrained("microsoft/TRELLIS.2")
t2_pipe.to(DEVICE)

def trellis2_job(item: dict) -> str:
    name = item.get("name", "character")
    workdir = os.path.join(OUTPUT_DIR, name)
    os.makedirs(workdir, exist_ok=True)
    kwargs = dict(num_steps=12, guidance_scale=4.5)
    if item.get("prompt"):
        kwargs["prompt"] = enhance_prompt(item["prompt"])
    if item.get("image"):
        kwargs["reference_image"] = Image.open(item["image"]).convert("RGB")
    print(f"🔨 TRELLIS.2: {name} ...")
    with torch.no_grad():
        result = t2_pipe(**kwargs)
    raw = os.path.join(workdir, "raw.glb")
    result.mesh.export(raw)  # trimesh → GLB
    final = os.path.join(workdir, f"{name}.glb")
    postprocess_glb(raw, final, TARGET_FACES)
    if DRIVE_OK:
        shutil.copy2(final, os.path.join(DRIVE_FOLDER, f"{name}.glb"))
    try:
        from google.colab import files as _cf
        _cf.download(final)
    except ImportError:
        pass
    return final

for job in jobs:
    trellis2_job(job)
print("\n✅ TRELLIS.2 batch done")
